# Impulse and Torque MLP (vertex-transform, no GNN)

This notebook trains a physics-structured and physics-informed MLP for a
rigid body interacting with a plane.

**Variant:** the GCN / GNN is removed. We still place the body's mesh
vertices in world space (rotation + translation, plus per-vertex velocity
from rigid-body kinematics), but those transformed vertices are flattened
and fed **directly into the MLP** — no graph convolutions, no `edge_index`,
no `torch_geometric`.

As input we get the rotation as a 6-D rotation matrix (with sin and cos
of roll / pitch / yaw). We parameterise normal force with Hooke, similar
to our simulations. We internally predict the contact normal. We couple
torque to force via cross product: `torque = r_lever x f`. In the loss we
use a Huber loss with a weight for energy conservation.


This notebook trains a physics-structured and physics informed MLP for a rigid body interacting with a plane-
As input we get the rotation as a 6-D rotation Matrix (with cos and sin)
We parameterise normal force with Hooke, similar to our simulations
We internally predict the contact normal
We couple torque to force via cross product: torque = r_{lever} x f
In the loss we use a Hube loss with a weight for energy conservation

---

**Update — closing the gap to the GNN.** The earlier version of this MLP trained much worse than the GCN variant for two reasons, both now fixed in the `VertexMLPWrench` cell:

1. *Vertex features were crippled.* The GCN feeds each vertex 16 geometric channels (local xyz, world xyz, world z, body linear & angular velocity, and the per-vertex world velocity `v_lin + ω×r`). The MLP was keeping only the 3 world coordinates and silently dropping all velocity information. The full 16-D set is now built per vertex, matching the GCN.

2. *Flatten → pool.* The MLP concatenated a per-sample-variable selection of vertices into one flat vector, so a given physical vertex landed in a different slot on every sample — an alignment the network cannot learn. It now applies a shared per-vertex encoder followed by a permutation-invariant max+mean pool, exactly the reduction the GCN uses after message passing. This is the dominant fix.

The mesh is now decimated to `face_count=1000` to match the GCN notebook.

In [1]:
%pip install trimesh
%pip install fast-simplification

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import json
import math
from tqdm import tqdm
import trimesh

In [3]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

Using device: cpu


## Constants

In [4]:
TRAIN_TEST_SPLIT = 0.8

## Colab Only - Download data

In [5]:
def is_colab():
    try:
        import google.colab
        return True
    except Exception as e:
        return False

if is_colab():
    from google.colab import drive
    from tqdm import tqdm
    import os
    import json
    import shutil

    drive.mount('/content/drive', force_remount=True)

    # --- Copy the JSON file ---
    src_path = "/content/drive/MyDrive/final_output_contact_points.json"
    dst_path = "/content/final_output_contact_points.json"
    chunk_size = 1024 * 1024  # 1 MB
    file_size = os.path.getsize(src_path)

    with open(src_path, 'rb') as src, open(dst_path, 'wb') as dst:
        with tqdm(total=file_size, unit='B', unit_scale=True, desc="Copying JSON to /content") as pbar:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))
    print("Done! File is now in:", dst_path)

    # --- Copy the blender_models folder ---
    src_folder = "/content/drive/MyDrive/blender_models"
    dst_folder = "/content/blender_models"

    # Gather all files first so we know the total size for the progress bar
    all_files = []
    total_size = 0
    for root, dirs, files in os.walk(src_folder):
        for f in files:
            full = os.path.join(root, f)
            try:
                size = os.path.getsize(full)
            except OSError:
                size = 0
            all_files.append((full, size))
            total_size += size

    print(f"\nFound {len(all_files)} files in blender_models ({total_size / (1024**2):.1f} MB total)")

    os.makedirs(dst_folder, exist_ok=True)

    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Copying blender_models") as pbar:
        for src_file, size in all_files:
            rel = os.path.relpath(src_file, src_folder)
            dst_file = os.path.join(dst_folder, rel)
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)

            with open(src_file, 'rb') as src, open(dst_file, 'wb') as dst:
                while True:
                    chunk = src.read(chunk_size)
                    if not chunk:
                        break
                    dst.write(chunk)
                    pbar.update(len(chunk))

    print("Done! Folder is now in:", dst_folder)

    # --- Print the first entry of the JSON ---
    with open(dst_path, 'r') as f:
        data = json.load(f)
    first = data[0] if isinstance(data, list) else next(iter(data.values()))
    print("\nFirst entry:")
    print(json.dumps(first, indent=2))

## Dataset

Here we get the contat points with individual forces. From these forces, we calculate the torque and sum up the forces and torques to one force and one torque

In [6]:
class ContactDataset(Dataset):
    """World-frame wrench dataset for a rigid body on a plane.

    Features (13-D):
        [v_x, v_y, v_z,                           # linear velocity (world frame)
         w_x, w_y, w_z,                           # angular velocity (world frame)
         rel_pos_z,                               # height above plane
         sin(roll), cos(roll),
         sin(pitch), cos(pitch),
         sin(yaw), cos(yaw)]

    Per-sample extras (used by the GCN to place vertices in world space):
        self_position: (3,) world-frame position of the body origin (= COM
                       used in the torque computation: lever = r_world - cube_pos).

    Targets (world frame, PHYSICAL UNITS — not normalised):
        force:  (3,)   sum of per-contact forces
        torque: (3,)   sum of (r_world - com_world) x f_world
    """

    def __init__(self, data_list):
        features, forces, torques, collisions = [], [], [], []
        lin_vels, ang_vels, self_positions = [], [], []

        for contact in data_list:
            rel_pos = contact["relative_position_to_collider"]
            rel_rot = contact["relative_rotation_to_collider"]
            lin_vel = contact["linear_velocity"]
            ang_vel = contact["angular_velocity"]
            cube_pos = contact["self_position"]

            roll, pitch, yaw = rel_rot["roll"], rel_rot["pitch"], rel_rot["yaw"]
            feats = np.array([
                lin_vel["x"], lin_vel["y"], lin_vel["z"],
                ang_vel["x"], ang_vel["y"], ang_vel["z"],
                rel_pos["z"],
                np.sin(roll), np.cos(roll),
                np.sin(pitch), np.cos(pitch),
                np.sin(yaw), np.cos(yaw),
            ], dtype=np.float32)

            cube_pos_numpy = np.array([cube_pos["x"], cube_pos["y"], cube_pos["z"]],
                                      dtype=np.float32)
            force_numpy = np.zeros(3, dtype=np.float32)
            torque_numpy = np.zeros(3, dtype=np.float32)

            for p in contact.get("points", []):
                p_force = p["force"]
                lever_pos = p["contact_position_world"]
                point_force_numpy = np.array(
                    [p_force["x"], p_force["y"], p_force["z"]], dtype=np.float32)
                lever_rel_pos = np.array(
                    [lever_pos["x"], lever_pos["y"], lever_pos["z"]],
                    dtype=np.float32) - cube_pos_numpy

                force_numpy += point_force_numpy
                torque_numpy += np.cross(lever_rel_pos, point_force_numpy)

            features.append(feats)
            forces.append(force_numpy)
            torques.append(torque_numpy)
            collisions.append([min(len(contact.get("points", [])), 1)])
            lin_vels.append([lin_vel["x"], lin_vel["y"], lin_vel["z"]])
            ang_vels.append([ang_vel["x"], ang_vel["y"], ang_vel["z"]])
            self_positions.append(cube_pos_numpy)

        self.features       = torch.FloatTensor(np.asarray(features))
        self.forces         = torch.FloatTensor(np.asarray(forces))
        self.torques        = torch.FloatTensor(np.asarray(torques))
        self.collisions     = torch.FloatTensor(np.asarray(collisions))
        self.lin_vels       = torch.FloatTensor(np.asarray(lin_vels, dtype=np.float32))
        self.ang_vels       = torch.FloatTensor(np.asarray(ang_vels, dtype=np.float32))
        self.self_positions = torch.FloatTensor(np.asarray(self_positions, dtype=np.float32))

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (
            self.features[idx],
            {
                "force":         self.forces[idx],
                "torque":        self.torques[idx],
                "is_collision":  self.collisions[idx],
                "lin_vel":       self.lin_vels[idx],
                "ang_vel":       self.ang_vels[idx],
                "self_position": self.self_positions[idx],
            },
        )

### Show dataset

show the first entry of the dataset

In [7]:
json_file = 'final_output_contact_points.json'
with open(json_file, 'r') as f:
    data = json.load(f)
if isinstance(data, dict):
    data = [data]

full_dataset = ContactDataset(data)


Print the first entry of the dataset

In [8]:
print(full_dataset[0])

(tensor([-5.7540e-11,  5.8263e-11,  3.8904e-02, -7.6838e-03,  8.6005e-02,
         1.9989e-02,  3.0494e-01, -3.3218e-01, -9.4322e-01,  2.0897e-01,
         9.7792e-01, -2.9413e-01,  9.5577e-01]), {'force': tensor([-9.1224e-13,  6.0816e-12,  2.8818e-01]), 'torque': tensor([ 8.8943e-03, -3.1792e-02,  6.9908e-13]), 'is_collision': tensor([1.]), 'lin_vel': tensor([-5.7540e-11,  5.8263e-11,  3.8904e-02]), 'ang_vel': tensor([-0.0077,  0.0860,  0.0200]), 'self_position': tensor([-1.3479e-10, -9.2604e-11,  3.0494e-01])})


Showing some info of the dataset

In [9]:
print("collisions:", int(full_dataset.collisions.sum().item()),
      "/", len(full_dataset))
mask = full_dataset.collisions.squeeze(-1).bool()
if mask.any():
    f_rms = full_dataset.forces[mask].pow(2).mean().sqrt().item()
    t_rms = full_dataset.torques[mask].pow(2).mean().sqrt().item()
    print(f"\nContact-only RMS force  = {f_rms:.4f}")
    print(f"Contact-only RMS torque = {t_rms:.4f}")
    print(f"Suggested w_torque / w_force ratio ≈ {f_rms / max(t_rms, 1e-8):.3f}")


collisions: 1024037 / 1958062

Contact-only RMS force  = 1064.2872
Contact-only RMS torque = 202.8762
Suggested w_torque / w_force ratio ≈ 5.246


Split the dataset into train and test dataset and create data loaders

In [10]:
train_size = int(TRAIN_TEST_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator = torch.Generator())

use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=4 if use_gpu else 0,
    pin_memory=False,
    persistent_workers=False,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=4 if use_gpu else 0,
    pin_memory=False,
    persistent_workers=False,
)

Validation checks on the dataset:

In [11]:
input_dim = full_dataset.features.shape[1]
print(f"input_dim={input_dim}  (expect 10)")
print(f"force target shape = {tuple(full_dataset.forces.shape)}  (expect (N, 3))")
print(f"torque target shape = {tuple(full_dataset.torques.shape)}  (expect (N, 3))")
assert max(full_dataset.collisions) == 1


input_dim=13  (expect 10)
force target shape = (1958062, 3)  (expect (N, 3))
torque target shape = (1958062, 3)  (expect (N, 3))


## Mesh (Sphere trees)

We load the mesh and keep only the **vertex positions in the body's local
frame**. There is no graph here — no edges, no neighbours — because the
MLP doesn't operate on a graph. The vertices are a fixed-size, fixed-order
geometric description of the body's shape, and we feed them into the MLP
after rotating them into world space.


In [12]:
import numpy as np

with open("../sphere_trees/bunny.spheres") as sphere_file:
    sphere_lines = sphere_file.readlines()

sphere_list = []
for sphere_line in sphere_lines:
    parts = sphere_line.split()  # no arg: handles multiple spaces + strips \n
    if not parts:
        continue  # skip empty lines
    try:
        sphere_list.append([float(v) for v in parts])
    except ValueError:
        print(f"failed parsing line: {sphere_line!r}")

spheres = np.array(sphere_list)
num_spheres = spheres.shape[0]
print(spheres.shape)
print(spheres[:10])

(100000, 4)
[[ 0.15473682 -0.24821055  0.10168421  0.31029269]
 [-0.34168422 -0.09663159  0.11431581  0.20863426]
 [-0.43010527  0.22294736  0.18252629  0.12969711]
 [-0.19389474 -0.47052634  0.03978944  0.10761782]
 [-0.18505263 -0.49705267  0.23305261  0.08765664]
 [-0.20778948 -0.3303158   0.22168422  0.08025049]
 [-0.11431581 -0.00189477 -0.03600001  0.07955279]
 [-0.43263161  0.08526313  0.33915788  0.07886076]
 [ 0.50589478 -0.40989476  0.13957894  0.07815155]
 [-0.20526317 -0.30000001 -0.03473687  0.07788653]]


In [13]:
print(sphere_list[:10])

[[0.154736817, -0.248210549, 0.101684213, 0.310292691], [-0.341684222, -0.0966315866, 0.114315808, 0.208634257], [-0.430105269, 0.222947359, 0.18252629, 0.129697114], [-0.193894744, -0.470526338, 0.0397894382, 0.107617818], [-0.185052633, -0.49705267, 0.233052611, 0.0876566395], [-0.207789481, -0.330315799, 0.221684217, 0.0802504867], [-0.114315808, -0.00189477205, -0.0360000134, 0.079552792], [-0.432631612, 0.085263133, 0.339157879, 0.0788607597], [0.50589478, -0.409894764, 0.139578938, 0.0781515539], [-0.205263168, -0.300000012, -0.0347368717, 0.0778865293]]


## Model

Physically structured MLP. The vertices are rotated (and translated) into
world space, the per-vertex world-frame velocity is computed, and then
**everything is flattened and fed into the MLP** — no graph convolutions,
no message passing.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

HEAD_OUT_DIM = 11  # 1 (collision) + 1 (depth) + 3 (force_residual) + 3 (normal) + 3 (lever)

LIN_VEL_SLICE = slice(0, 3)
ANG_VEL_SLICE = slice(3, 6)


class ResBlock(nn.Module):
    def __init__(self, width, expansion=4):
        super().__init__()
        self.act = nn.ReLU(inplace=True)

        self.block = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, width * expansion),
            self.act,
            nn.LayerNorm(width * expansion),
            nn.Linear(width * expansion, width),
            self.act,
        )

    def forward(self, x):
        return self.act(x + self.block(x))


class WrenchPredictor(nn.Module):
    """
    Predicts net wrench (force + torque) with a head whose force assembly
    matches the physics sim, plus a learned residual to absorb deviations
    the analytical model cannot express.  (Unchanged from the vertex version.)
    """

    def __init__(self, input_dim=13, width=256, num_blocks=5,
                 baseline_k=1e3, learn_k=True,
                 baseline_bounciness=0.5, learn_bounciness=True,
                 baseline_mass=1.0, learn_mass=True,
                 head_hidden=64):
        super().__init__()

        # --- backbone ---
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.LayerNorm(width),
            nn.GELU(),
        )

        backbone_layers = [nn.LayerNorm(width)]
        # NB: range(num_blocks, 1, -1) builds num_blocks - 1 ResBlocks.
        for _ in range(num_blocks, 1, -1):
            backbone_layers.append(ResBlock(width))
        self.backbone = nn.Sequential(*backbone_layers)

        self.head_trunk = nn.Sequential(
            nn.Linear(width, head_hidden),
            nn.LayerNorm(head_hidden),
            nn.GELU(),
        )
        self.head_out = nn.Linear(head_hidden, HEAD_OUT_DIM)

        # Physics parameters.
        self.k = nn.Parameter(
            torch.tensor(baseline_k, dtype=torch.float32),
            requires_grad=learn_k,
        )
        self.bounciness = nn.Parameter(
            torch.tensor(baseline_bounciness, dtype=torch.float32),
            requires_grad=learn_bounciness,
        )
        self.mass = nn.Parameter(
            torch.tensor(baseline_mass, dtype=torch.float32),
            requires_grad=learn_mass,
        )

    def forward(self, x, velocity):
        h = self.input_proj(x)
        h = self.backbone(h)

        raw = self.head_out(self.head_trunk(h))
        collision_logit, depth_raw, force_residual, normal_raw, lever = raw.split(
            [1, 1, 3, 3, 3], dim=-1
        )

        k_pos    = F.softplus(self.k)     if self.k.requires_grad else self.k
        mass_pos = F.softplus(self.mass)  if self.mass.requires_grad else self.mass
        mass_pos = torch.clamp(mass_pos, min=1e-6)
        bounciness = torch.sigmoid(self.bounciness)

        c = 2.0 * torch.sqrt(k_pos * mass_pos) * bounciness

        depth = torch.clamp(depth_raw, max=0.0)
        contact_normal = F.normalize(normal_raw, dim=-1, eps=1e-8)

        F_spring = -k_pos * depth
        vel_normal = (velocity * contact_normal).sum(dim=-1, keepdim=True)
        F_damping = -c * vel_normal
        F_mag = F.relu(F_spring + F_damping)

        force_normal = F_mag * contact_normal
        force_vec    = force_normal + force_residual
        torque_vec   = torch.cross(lever, force_vec, dim=-1)

        return {
            "collision_logit": collision_logit,
            "force":  force_vec,
            "torque": torque_vec,
            "aux": {
                "contact_normal": contact_normal,
                "lever":          lever,
                "depth":          depth,
                "k":              k_pos.detach(),
                "c":              c.detach(),
                "mass":           mass_pos.detach(),
                "F_mag":          F_mag,
                "F_spring":       F_spring,
                "F_damping":      F_damping,
                "force_normal":   force_normal,
                "force_residual": force_residual,
            },
        }


# --------------------------------------------------------------------------
# Rotation from the state features (unchanged)
# --------------------------------------------------------------------------
def _rotmat_from_sincos(features: torch.Tensor) -> torch.Tensor:
    """[B, F] feature batch -> [B, 3, 3] rotation matrix.

    Z-Y-X intrinsic: R = Rz(yaw) Ry(pitch) Rx(roll).
    """
    sr, cr = features[:, 7:8],  features[:, 8:9]
    sp, cp = features[:, 9:10], features[:, 10:11]
    sy, cy = features[:, 11:12], features[:, 12:13]

    row0 = torch.cat([cy * cp,  cy * sp * sr - sy * cr,  cy * sp * cr + sy * sr], dim=-1)
    row1 = torch.cat([sy * cp,  sy * sp * sr + cy * cr,  sy * sp * cr - cy * sr], dim=-1)
    row2 = torch.cat([-sp,      cp * sr,                 cp * cr               ], dim=-1)
    return torch.stack([row0, row1, row2], dim=1)


# --------------------------------------------------------------------------
# Sphere-packing utilities
# --------------------------------------------------------------------------
def _gather_nodes(t: torch.Tensor, order: torch.Tensor) -> torch.Tensor:
    """Gather [B, N, C] along dim 1 with an index expanded to t's own channel
    width C.  (A single hardcoded expand breaks as soon as tensors of
    different widths -- 3D positions vs. 1D radii -- are gathered.)"""
    idx = order.unsqueeze(-1).expand(-1, -1, t.size(-1))
    return torch.gather(t, 1, idx)


def load_spheres(path: str) -> torch.Tensor:
    """Load an [N, 4] float32 tensor of (cx, cy, cz, r) from a whitespace-
    separated text file, one sphere per line.  Lines that don't parse to at
    least 4 floats (headers, counts, comments) are skipped.

    NOTE: verify the column order of your file (e.g. CGVR .spheres) before
    trusting this -- open it and check whether the radius is really the
    4th column.
    """
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            try:
                vals = [float(p) for p in parts]
            except ValueError:
                continue
            if len(vals) >= 4:
                rows.append(vals[:4])
    if not rows:
        raise ValueError(f"No sphere rows found in {path}")
    return torch.tensor(rows, dtype=torch.float32)


# --------------------------------------------------------------------------
# Sphere-packing wrench model
# --------------------------------------------------------------------------
class SphereMLPWrench(nn.Module):
    """Wrench prediction from a sphere packing (ProtoSphere-style) instead of
    mesh vertices.

    Geometry input is [N, 4]: (cx, cy, cz, r) in the *body frame*.
    Only the centers are rotated/translated into the world frame; the radius
    is invariant under rigid motions and is passed through as a per-sphere
    scalar feature.
    """

    # Per-sphere feature layout:
    #   c_local (3) | c_world (3) | radius (1) | clearance (1)
    #   v_center (3) | v_lin (3) | v_ang (3)
    SPHERE_GEOM_DIM = 17

    def __init__(self, state_dim: int = 13,
                 spheres_per_graph: int = 4000,
                 width: int = 256, num_blocks: int = 5,
                 encoder_dim: int = 256):
        super().__init__()
        self.N = spheres_per_graph
        self.state_dim = state_dim
        self.encoder_dim = encoder_dim

        # Encoder over the flattened all-sphere feature vector.
        flat_dim = self.N * self.SPHERE_GEOM_DIM
        self.sphere_encoder = nn.Sequential(
            nn.Linear(flat_dim, encoder_dim),
            nn.LayerNorm(encoder_dim),
            nn.GELU(),
            ResBlock(encoder_dim),
        )

        self.head = WrenchPredictor(
            input_dim=state_dim + encoder_dim,
            width=width, num_blocks=num_blocks,
        )

    def forward(self, features, spheres, body_position=None):
        """
        features:      [B, state_dim]  rigid body state
        spheres:       [N, 4]          body-frame packing, columns (cx, cy, cz, r)
        body_position: [B, 3]          world position of the body origin (optional)
        """
        B = features.size(0)
        N = spheres.size(0)
        if N < self.N:
            raise ValueError(
                f"Packing has {N} spheres but the model expects >= {self.N}. "
                f"Lower spheres_per_graph (or pad the packing)."
            )

        centers = spheres[:, :3]                                   # [N, 3]
        radii   = spheres[:, 3:4]                                  # [N, 1]

        R = _rotmat_from_sincos(features)                          # [B, 3, 3]

        # Rotate ONLY the centers -- the radius must never see R.
        c_local = centers.unsqueeze(0).expand(B, N, 3)
        arm = torch.einsum("bij,bnj->bni", R, c_local)             # offset from body origin, world frame
        if body_position is not None:
            c_world = arm + body_position.unsqueeze(1)
        else:
            c_world = arm

        r = radii.unsqueeze(0).expand(B, N, 1)                     # invariant per-sphere scalar

        v_lin = features[:, LIN_VEL_SLICE].unsqueeze(1).expand(B, N, 3)
        v_ang = features[:, ANG_VEL_SLICE].unsqueeze(1).expand(B, N, 3)

        # Signed distance of each sphere's SURFACE to the ground plane z = 0.
        # clearance < 0  <=>  the sphere penetrates the ground.
        clearance = c_world[..., 2:3] - r                          # [B, N, 1]

        # Keep the K spheres whose surface is closest to the ground.
        # Sorting by center height would miss a big sphere that touches first;
        # topk with fixed K keeps shapes static (CUDA-graph friendly) and
        # yields a canonical, deepest-first ordering for the flat encoder.
        _, order = torch.topk(clearance.squeeze(-1), self.N,
                              dim=-1, largest=False)               # [B, K]
        c_local, c_world, arm, r, clearance, v_lin, v_ang = (
            _gather_nodes(t, order)
            for t in (c_local, c_world, arm, r, clearance, v_lin, v_ang)
        )

        # Velocity of each sphere center: v + w x arm.
        # Use the arm (R @ c_local), NOT the absolute world position --
        # otherwise the lever is wrong as soon as body_position != 0.
        v_center = v_lin + torch.cross(v_ang, arm, dim=-1)

        sphere_feats = torch.cat(
            [c_local, c_world, r, clearance, v_center, v_lin, v_ang], dim=-1
        )                                                          # [B, K, 17]

        flat = sphere_feats.reshape(B, self.N * self.SPHERE_GEOM_DIM)
        embed = self.sphere_encoder(flat)                          # [B, encoder_dim]

        x = torch.cat([features, embed], dim=-1)
        out = self.head(x, velocity=features[:, LIN_VEL_SLICE])

        # Geometric penetration of the deepest sphere. Free supervision /
        # consistency target for the head's learned `depth` output:
        #   L_depth = |depth - clamp(min_clearance, max=0)|
        out["aux"]["min_clearance"] = clearance.min(dim=1).values  # [B, 1]
        return out


def make_fast_predictor(input_dim=13, width=256, num_blocks=5,
                        encoder_dim=256, spheres_per_graph=10000):
    return SphereMLPWrench(
        state_dim=input_dim,
        spheres_per_graph=spheres_per_graph,
        width=width, num_blocks=num_blocks,
        encoder_dim=encoder_dim,
    )

In [15]:
model = make_fast_predictor().to(device)

Print the model

In [16]:

print(f"Model: {model}")
print(f"Backbone: {model.head.backbone}")
print(f"Head trunk: {model.head.head_trunk}")
print(f"Head out: {model.head.head_out}")

Model: SphereMLPWrench(
  (sphere_encoder): Sequential(
    (0): Linear(in_features=68000, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): GELU(approximate='none')
    (3): ResBlock(
      (act): ReLU(inplace=True)
      (block): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (1): Linear(in_features=256, out_features=1024, bias=True)
        (2): ReLU(inplace=True)
        (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (4): Linear(in_features=1024, out_features=256, bias=True)
        (5): ReLU(inplace=True)
      )
    )
  )
  (head): WrenchPredictor(
    (input_proj): Sequential(
      (0): Linear(in_features=269, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (2): GELU(approximate='none')
    )
    (backbone): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_af

Check the model

In [17]:
cpu_model = make_fast_predictor()
features = torch.zeros([1, 13])
vp = torch.tensor(spheres, dtype=torch.float32)
out = cpu_model(features, spheres=vp)


In [18]:
model.eval()
with torch.no_grad():
    # Single-sample sanity check: 1 sample of 13-D state.
    features = torch.zeros([1, 13], device=device)
    vp = torch.tensor(spheres, dtype=torch.float32, device=device)
    out = model(features, spheres=vp)
    print("force shape :", tuple(out["force"].shape))
    print("torque shape:", tuple(out["torque"].shape))


force shape : (1, 3)
torque shape: (1, 3)


### Benchmarking

Benchmark the evaluation speed of the model

In [19]:
NUM_BENCHMARKS = 1000
import time
import torch._logging
torch._logging.set_logs(recompiles=True, graph_breaks=True)

features = torch.rand((1, 13), device=device)
vp = torch.tensor(spheres, dtype=torch.float32, device=device)

results = []
benchmark_model = make_fast_predictor().to(device).eval()
benchmark_model = torch.compile(benchmark_model)
benchmark_model.eval()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

    with torch.inference_mode():
        # warmup
        for _ in tqdm(range(1000), desc="Warmup"):
            benchmark_model(features, spheres=vp)
        print("=====Warmup completed======")

        for _ in tqdm(range(NUM_BENCHMARKS), desc="Benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            benchmark_model(features, spheres=vp)
            torch.cuda.synchronize()
            end = time.perf_counter()
            results.append((end - start) * 1000)

    torch.backends.cudnn.benchmark = False
    per_pred_ms = sum(results) / NUM_BENCHMARKS
    print(f"Per prediction: {per_pred_ms:.2f} ms")


In [20]:
if torch.cuda.is_available():
    print(f"\n=== Benchmark Results ({device}) ===")
    print(f"Max: {max(results)} ms")
    print(f"Min: {min(results)} ms")
    print(f"Avg: {sum(results) / len(results)} ms")
    print(f"Median: {sorted(results)[len(results) // 2]} ms")
    print("Results:")
    print(results)


### CUDA Graph Benchmark

Captures the forward pass as a CUDA graph so the ~40 per-kernel dispatches are replaced by a single `cuGraphLaunch` on every call.

In [21]:
if torch.cuda.is_available():
    # Fixed batch size for the captured graph.
    B = 1

    cuda_graph_model = make_fast_predictor().to(device).eval()

    # Static tensors — these addresses are baked into the captured graph.
    # Shapes must exactly match what the model is replayed with later.
    static_features   = torch.zeros(B, 13, device=device)
    static_vertex_pos = torch.tensor(spheres, dtype=torch.float32,
                                     device=device)

    # --- Warmup before capture — must run on a side stream ---
    warmup_stream = torch.cuda.Stream()
    warmup_stream.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(warmup_stream):
        for _ in tqdm(range(1000), desc="CUDA graph warmup"):
            with torch.inference_mode():
                _ = cuda_graph_model(static_features, spheres=static_vertex_pos)
    torch.cuda.current_stream().wait_stream(warmup_stream)
    torch.cuda.synchronize()

    # --- Capture ---
    cuda_graph = torch.cuda.CUDAGraph()
    with torch.inference_mode(), torch.cuda.graph(cuda_graph):
        static_output = cuda_graph_model(static_features, spheres=static_vertex_pos)
    print("CUDA graph captured.")

    # --- Benchmark ---
    # Pre-generate inputs OUTSIDE the timing loop.
    inputs = [torch.rand(B, 13, device=device) for _ in range(NUM_BENCHMARKS)]
    torch.cuda.synchronize()

    cuda_graph_results = []
    with torch.inference_mode():
        for x in tqdm(inputs, desc="CUDA graph benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            static_features.copy_(x)
            cuda_graph.replay()
            torch.cuda.synchronize()
            end = time.perf_counter()
            cuda_graph_results.append((end - start) * 1000)

    print(f"\n=== CUDA Graph Benchmark Results ({device}) ===")
    print(f"Max:    {max(cuda_graph_results):.4f} ms")
    print(f"Min:    {min(cuda_graph_results):.4f} ms")
    print(f"Avg:    {sum(cuda_graph_results) / len(cuda_graph_results):.4f} ms")
    print(f"Median: {sorted(cuda_graph_results)[len(cuda_graph_results) // 2]:.4f} ms")
    print(f"\nSpeedup over baseline: "
          f"{(sum(results) / len(results)) / (sum(cuda_graph_results) / len(cuda_graph_results)):.1f}x")


## 4. Loss

Huber (smooth-L1) loss replaces L1.  Near zero it's quadratic (smooth gradients,
won't over-punish tiny residuals); far from zero it's linear (robust to the
occasional outlier).  `delta=1.0` is in *normalized* target units so it's
roughly one standard deviation of the target.

**Energy-conservation penalty.**  For each collision sample we integrate one
timestep forward using the *predicted* wrench and check whether the body's
kinetic energy would grow beyond `e^2 * KE_before` (where `e` is the
coefficient of restitution).  Any excess is squared and added to the loss,
so the term is zero for physically admissible predictions and grows smoothly
when the model would inject energy — the exact failure mode you were seeing
at low collision speeds.  You control it with `w_energy`, `dt`, `mass`,
`inertia_diag`, and `restitution` when constructing `WrenchLoss`.


In [22]:
class WrenchLoss(nn.Module):
    """Physics-structured loss for a Hooke (linear elastic) contact model.

    Components:
        - BCE on collision_logit (with pos_weight for class imbalance).
        - Masked Huber on force and torque, operating in PHYSICAL units.
        - Soft penalty on f_n < 0 (Signorini violation).
        - Soft penalty on energy gain during collision (restitution-aware),
          using a BOUNDED log-based term so large violations at init don't
          produce enormous gradients that kill regression learning.

    Energy-conservation term
    ------------------------
    Given a predicted force F and torque tau applied over one timestep dt to a
    body with mass m and (diagonal) inertia I, the post-step velocities are
        v'   = v   + (F   / m) * dt
        w'   = w   + (I^-1 tau) * dt
    and the kinetic energy is  KE = 0.5 m |v|^2 + 0.5 w^T I w.
    For a real collision with coefficient of restitution e in [0, 1] we expect
        KE'  <=  e^2 * KE_before.
    We define the energy ratio
        r = KE' / (e^2 * KE_before)
    and penalise log(max(r, 1))^2. This is zero when r <= 1 (admissible),
    grows like (log r)^2 when r > 1, and its gradient in r is bounded — so a
    random-init network that predicts wildly wrong forces at epoch 0 won't
    produce an exploding energy gradient that pushes the model into the
    degenerate F≈0 basin.

    Warmup: w_energy typically starts at 0 and is ramped up over several
    epochs by the training loop via `set_energy_weight`, so the regression
    heads (force, torque) get to learn first before conservation pressure
    kicks in.

    Because targets are NOT pre-normalised, you may need to set w_force and
    w_torque to bring the two regression terms to comparable magnitude. A good
    heuristic is to set:
        w_force  ~ 1 / (force_rms_in_physical_units)
        w_torque ~ 1 / (torque_rms_in_physical_units)
    so both contribute roughly equally early in training.
    """
    def __init__(self,
                 w_force=1.0, w_torque=1.0, w_collision=1.0,
                 w_energy=0.0,
                 huber_delta=1.0, pos_weight=None,
                 dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                 restitution=0.5):
        super().__init__()
        self.w_force     = w_force
        self.w_torque    = w_torque
        self.w_collision = w_collision
        self.w_energy    = w_energy
        self.huber_delta = huber_delta
        self.dt          = float(dt)
        self.mass        = float(mass)
        self.restitution = float(restitution)
        # Inertia tensor (diagonal) for a unit cube by default: I = (1/6) m a^2
        # with m=1, a=1. Override via the constructor to match your simulated body.
        self.register_buffer(
            "inertia_diag",
            torch.tensor(inertia_diag, dtype=torch.float32),
        )
        # pos_weight is a tensor; register as buffer so .to(device) moves it.
        if pos_weight is not None and not torch.is_tensor(pos_weight):
            pos_weight = torch.tensor(float(pos_weight))
        self.register_buffer(
            "pos_weight",
            pos_weight if pos_weight is not None else torch.tensor(1.0),
        )
        self._has_pos_weight = pos_weight is not None

    def set_energy_weight(self, w):
        """Runtime hook for the training loop's warmup schedule."""
        self.w_energy = float(w)

    def _kinetic_energy(self, v, w):
        """KE = 0.5 m |v|^2 + 0.5 w^T I w for diagonal I. Shapes: (B,3)."""
        ke_lin = 0.5 * self.mass * (v * v).sum(dim=-1, keepdim=True)
        ke_rot = 0.5 * (self.inertia_diag * w * w).sum(dim=-1, keepdim=True)
        return ke_lin + ke_rot

    def forward(self, preds, targets):
        mask   = targets["is_collision"]                   # (B, 1)
        n_coll = mask.sum().clamp_min(1.0)

        # --- collision BCE ---
        loss_collision = F.binary_cross_entropy_with_logits(
            preds["collision_logit"], mask.float(),
            pos_weight=self.pos_weight if self._has_pos_weight else None,
            reduction="mean",
        )
        # --- masked Huber on force and torque (physical units) ---
        raw_f = F.huber_loss(preds["force"],  targets["force"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        raw_t = F.huber_loss(preds["torque"], targets["torque"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        loss_force  = (raw_f.sum(dim=-1, keepdim=True) * mask).sum() / n_coll
        loss_torque = (raw_t.sum(dim=-1, keepdim=True) * mask).sum() / n_coll

        # --- energy-conservation penalty (bounded, log-based) ---
        # Integrate one step using the predicted wrench and compare KE before vs after.
        # Only collision samples contribute (non-contact steps have F=tau=0 anyway).
        v = targets["lin_vel"]                              # (B, 3)
        w = targets["ang_vel"]                              # (B, 3)
        f_pred = preds["force"]                             # (B, 3)
        t_pred = preds["torque"]                            # (B, 3)

        v_next = v + (f_pred / self.mass) * self.dt
        # Diagonal inertia -> element-wise divide
        w_next = w + (t_pred / self.inertia_diag) * self.dt

        ke_before = self._kinetic_energy(v, w)              # (B, 1)
        ke_after  = self._kinetic_energy(v_next, w_next)    # (B, 1)

        # Log-ratio penalty:
        #   r = ke_after / (e^2 * ke_before + eps),  penalty = max(log r, 0)^2
        # Bounded gradient in F: d/dF log(ke_after) scales as 1/ke_after, so at
        # init where ke_after is huge the gradient is SMALL — the opposite of
        # (ke_after - budget)^2, which has gradient proportional to ke_after.
        eps       = 1e-6
        ke_budget = (self.restitution ** 2) * ke_before
        log_ratio = torch.log(ke_after + eps) - torch.log(ke_budget + eps)
        excess_log  = F.relu(log_ratio)                     # zero when admissible
        loss_energy = ((excess_log ** 2) * mask).sum() / n_coll

        total = (self.w_force     * loss_force
               + self.w_torque    * loss_torque
               + self.w_collision * loss_collision
               + 0   * loss_energy)

        # Diagnostic: fraction of collision samples that currently violate conservation,
        # plus the geometric-mean ratio so you can see HOW MUCH they violate by.
        with torch.no_grad():
            violating = ((excess_log > 0).float() * mask).sum() / n_coll
            # Mean log-ratio over collision samples (in log space so it's well-behaved)
            mean_log_ratio = (log_ratio * mask).sum() / n_coll

        return total, {
            "force":             loss_force.item(),
            "torque":            loss_torque.item(),
            "collision":         loss_collision.item(),
            "energy":            loss_energy.item(),
            "energy_violating":  violating.item(),
            "energy_mean_logr":  mean_log_ratio.item(),
            "w_energy":          self.w_energy,
            "total":             total.item(),
            "active_collisions": n_coll.item(),
            "k":                 preds["aux"]["k"].item(),
        }

## Training

In [23]:
def train_model(model, train_loader, val_loader, train_dataset,
                epochs=200, lr=1e-4, weight_decay=1e-4,
                w_force=1.0, w_torque=1.0, w_collision=0.5,
                # Energy-conservation warmup schedule.
                # Rationale: starting with w_energy>0 produces enormous gradients
                # at init (ke_after is huge for a random-init network) and pushes
                # the model into the degenerate F≈0 basin, where regression loss
                # plateaus at force_rms. Warming up from 0 lets the force/torque
                # heads learn a reasonable solution first; only then do we
                # gently tighten conservation.
                w_energy_max=0.1, energy_warmup_start=20, energy_warmup_epochs=30,
                dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                restitution=0.5):
    model.to(device)

    # Vertex positions are constant across all samples — build once.
    vp = torch.tensor(spheres, dtype=torch.float32, device=device)

    # Class-imbalance weight for BCE: #no-contact / #contact on train set.
    collisions = train_dataset.collisions.squeeze(-1).bool()
    n_pos = int(collisions.sum().item())
    n_neg = int((~collisions).sum().item())
    pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0
    print(f"BCE pos_weight = {pos_weight:.3f}  ({n_pos} contacts / {n_neg} non-contacts)")

    criterion = WrenchLoss(
        w_force=w_force, w_torque=w_torque, w_collision=w_collision,
        w_energy=0.0,  # ramped up by the warmup schedule below
        dt=dt, mass=mass, inertia_diag=inertia_diag, restitution=restitution,
        huber_delta=1.0, pos_weight=pos_weight,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=20
    )

    def energy_weight_at(epoch):
        """Linear ramp from 0 to w_energy_max over [start, start+epochs)."""
        if epoch < energy_warmup_start:
            return 0.0
        if energy_warmup_epochs <= 0:
            return float(w_energy_max)
        frac = (epoch - energy_warmup_start) / float(energy_warmup_epochs)
        return float(w_energy_max) * min(max(frac, 0.0), 1.0)

    best_val_loss = float('inf')
    loss_keys = ['total', 'force', 'torque', 'collision',
                 'energy', 'energy_violating', 'energy_mean_logr']

    for epoch in tqdm(range(epochs)):
        criterion.set_energy_weight(energy_weight_at(epoch))

        # --- Train ---
        model.train()
        train_losses = {k: 0.0 for k in loss_keys}
        for features, targets in train_loader:
            features = features.to(device)                       # [B, 13]
            targets  = {k: v.to(device) for k, v in targets.items()}
            out = model(
                features,
                spheres=vp,
                body_position=targets["self_position"],
            )
            optimizer.zero_grad()

            loss, components = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            for k in loss_keys:
                train_losses[k] += components[k]
        for k in loss_keys:
            train_losses[k] /= len(train_loader)

        # --- Validate ---
        model.eval()
        val_losses = {k: 0.0 for k in loss_keys}
        with torch.no_grad():
            for features, targets in val_loader:
                features = features.to(device)
                targets  = {k: v.to(device) for k, v in targets.items()}
                _, components = criterion(
                    model(
                        features,
                        spheres=vp,
                        body_position=targets["self_position"],
                    ),
                    targets,
                )
                for k in loss_keys:
                    val_losses[k] += components[k]
        for k in loss_keys:
            val_losses[k] /= len(val_loader)

        scheduler.step(val_losses['total'])

        if val_losses['total'] < best_val_loss:
            best_val_loss = val_losses['total']
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss':        best_val_loss,
            }, 'wrench_model_best.pth')

        if (epoch + 1) % 10 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            k_now  = components.get('k', float('nan'))
            we_now = criterion.w_energy
            print(f"Epoch {epoch+1}/{epochs} | LR: {lr_now:.2e} | k={k_now:.1f} | w_energy={we_now:.4f}")
            print(f"  Train: total={train_losses['total']:.4f} "
                  f"f={train_losses['force']:.4f} t={train_losses['torque']:.4f} "
                  f"c={train_losses['collision']:.4f} "
                  f"e={train_losses['energy']:.4f} vio={train_losses['energy_violating']:.2f} "
                  f"logr={train_losses['energy_mean_logr']:+.3f}")
            print(f"  Val:   total={val_losses['total']:.4f} "
                  f"f={val_losses['force']:.4f} t={val_losses['torque']:.4f} "
                  f"c={val_losses['collision']:.4f} "
                  f"e={val_losses['energy']:.4f} vio={val_losses['energy_violating']:.2f} "
                  f"logr={val_losses['energy_mean_logr']:+.3f}")

    return model


### Execute training

In [24]:
model = train_model(model, train_loader, val_loader, full_dataset,
                    epochs=200, lr=1e-3,
                    w_force=1.0, w_torque=1.0, w_collision=0.5)


BCE pos_weight = 0.912  (1024037 contacts / 934025 non-contacts)


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [03:10<?, ?it/s]


KeyboardInterrupt: 

## Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, loader, dataset, device="cuda",
             rel_floor_force=0.05, rel_floor_torque=0.05):
    """Evaluate in physical units (targets were not normalised).

    Args:
        dataset: the *underlying* ContactDataset (not a Subset). Kept for API
                 symmetry; no target stats are needed since targets are physical.
        rel_floor_{force,torque}: targets with magnitude below this (in
                 physical units) are excluded from the relative-error stats
                 to avoid division-by-near-zero blow-up.
    """
    model.eval()
    all_abs_f, all_abs_t = [], []
    all_rel_f, all_rel_t = [], []
    all_coll_correct     = []

    # Physics-diagnostic accumulators (frictionless model: only f_n sign check)
    n_fn_neg = 0
    n_total  = 0

    vp = torch.tensor(spheres, dtype=torch.float32, device=device)

    for features, targets in loader:
        features = features.to(device)
        f_tgt = targets["force"].to(device)
        t_tgt = targets["torque"].to(device)
        c_tgt = targets["is_collision"].to(device).squeeze(-1).bool()
        body_position = targets["self_position"].to(device)

        preds = model(features, spheres=vp, body_position=body_position)

        f_pred = preds["force"]
        t_pred = preds["torque"]
        c_pred = (torch.sigmoid(preds["collision_logit"]).squeeze(-1) > 0.5)

        all_coll_correct.append((c_pred == c_tgt).float().cpu())

        if c_tgt.any():
            f_pred_c = f_pred[c_tgt]
            f_tgt_c  = f_tgt[c_tgt]
            t_pred_c = t_pred[c_tgt]
            t_tgt_c  = t_tgt[c_tgt]

            abs_f = (f_pred_c - f_tgt_c).norm(dim=-1)
            abs_t = (t_pred_c - t_tgt_c).norm(dim=-1)
            all_abs_f.append(abs_f.cpu())
            all_abs_t.append(abs_t.cpu())

            f_norm = f_tgt_c.norm(dim=-1)
            t_norm = t_tgt_c.norm(dim=-1)
            mask_f = f_norm > rel_floor_force
            mask_t = t_norm > rel_floor_torque
            if mask_f.any():
                rel_f = (f_pred_c[mask_f] - f_tgt_c[mask_f]).norm(dim=-1) / f_norm[mask_f]
                all_rel_f.append(rel_f.cpu())
            if mask_t.any():
                rel_t = (t_pred_c[mask_t] - t_tgt_c[mask_t]).norm(dim=-1) / t_norm[mask_t]
                all_rel_t.append(rel_t.cpu())

            # Physics diagnostic: how often the Hooke-plus-residual allows f_n < 0.
            f_n_c = preds["aux"]["force_residual"][c_tgt]
            n_fn_neg += int((f_n_c < 0).sum().item())
            n_total  += int(c_tgt.sum().item())

    abs_f = torch.cat(all_abs_f) if all_abs_f else torch.empty(0)
    abs_t = torch.cat(all_abs_t) if all_abs_t else torch.empty(0)
    rel_f = torch.cat(all_rel_f) if all_rel_f else torch.empty(0)
    rel_t = torch.cat(all_rel_t) if all_rel_t else torch.empty(0)
    coll_acc = torch.cat(all_coll_correct).mean().item()

    def fmt_pct(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.1%}  p90={e.quantile(0.9):.1%}  "
                f"p99={e.quantile(0.99):.1%}")
    def fmt_abs(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.4f}  p90={e.quantile(0.9):.4f}  "
                f"p99={e.quantile(0.99):.4f}")

    print("─" * 60)
    print(f"Collision accuracy : {coll_acc:.3%}")
    print(f"Force  abs err     : {fmt_abs(abs_f)}")
    print(f"Torque abs err     : {fmt_abs(abs_t)}")
    print(f"Force  rel err     : {fmt_pct(rel_f)}  "
          f"(on {rel_f.numel()} / {abs_f.numel()} samples above floor)")
    print(f"Torque rel err     : {fmt_pct(rel_t)}  "
          f"(on {rel_t.numel()} / {abs_t.numel()} samples above floor)")
    if n_total > 0:
        print(f"Physics violations : f_n<0 in {n_fn_neg}/{n_total} "
              f"({100*n_fn_neg/n_total:.2f}%)")
    print("─" * 60)

    return {
        "collision_acc":  coll_acc,
        "force_abs_p50":  abs_f.median().item() if abs_f.numel() else float("nan"),
        "force_abs_p99":  abs_f.quantile(0.99).item() if abs_f.numel() else float("nan"),
        "torque_abs_p50": abs_t.median().item() if abs_t.numel() else float("nan"),
        "torque_abs_p99": abs_t.quantile(0.99).item() if abs_t.numel() else float("nan"),
        "fn_neg_rate":    (n_fn_neg / n_total) if n_total > 0 else float("nan"),
    }


## Print 100 data points

In [ ]:
@torch.no_grad()
def print_predictions(model, loader, n=100):
    """Print n predictions vs ground truth from the loader."""
    model.eval()
    all_f_pred, all_f_tgt = [], []
    all_t_pred, all_t_tgt = [], []
    all_coll_pred, all_coll_tgt = [], []

    vp = torch.tensor(spheres, dtype=torch.float32, device=device)

    for features, targets in loader:
        features = features.to(device)
        body_position = targets["self_position"].to(device)
        preds = model(features, spheres=vp, body_position=body_position)

        all_f_pred.append(preds["force"].cpu())
        all_t_pred.append(preds["torque"].cpu())
        all_f_tgt.append(targets["force"])
        all_t_tgt.append(targets["torque"])
        all_coll_pred.append(torch.sigmoid(preds["collision_logit"]).cpu())
        all_coll_tgt.append(targets["is_collision"])
        collected = sum(x.shape[0] for x in all_f_pred)
        if collected >= n:
            break

    f_pred = torch.cat(all_f_pred)[:n]
    f_tgt  = torch.cat(all_f_tgt)[:n]
    t_pred = torch.cat(all_t_pred)[:n]
    t_tgt  = torch.cat(all_t_tgt)[:n]
    c_pred = torch.cat(all_coll_pred)[:n].squeeze(-1)
    c_tgt  = torch.cat(all_coll_tgt)[:n].squeeze(-1)

    header = (f"{'#':>4s}  {'coll':>5s} {'pred':>5s}  "
              f"{'force_pred':>30s}  {'force_true':>30s}  "
              f"{'torque_pred':>30s}  {'torque_true':>30s}  ")
    print(header)
    print("─" * len(header))
    for i in range(n):
        cp = f"{c_pred[i]:.2f}"
        ct = f"{int(c_tgt[i].item())}"
        fp = f"[{f_pred[i,0]:8.3f}, {f_pred[i,1]:8.3f}, {f_pred[i,2]:8.3f}]"
        ft = f"[{f_tgt[i,0]:8.3f}, {f_tgt[i,1]:8.3f}, {f_tgt[i,2]:8.3f}]"
        tp = f"[{t_pred[i,0]:8.4f}, {t_pred[i,1]:8.4f}, {t_pred[i,2]:8.4f}]"
        tt = f"[{t_tgt[i,0]:8.4f}, {t_tgt[i,1]:8.4f}, {t_tgt[i,2]:8.4f}]"
        print(f"{i:4d}  {ct:>5s} {cp:>5s}  {fp:>30s}  {ft:>30s}  {tp:>30s}  {tt:>30s}")


print_predictions(model, val_loader, n=100)


   #   coll  pred                      force_pred                      force_true                     torque_pred                     torque_true  
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   0      0  0.00  [   0.003,    0.184,  -54.616]  [   0.000,    0.000,    0.000]  [  1.6209,   4.3160,   0.0147]  [  0.0000,   0.0000,   0.0000]
   1      1  0.99  [  -0.005,   -0.032,  120.788]  [  -0.000,    0.000,  120.035]  [-57.8874,  31.6484,   0.0059]  [-57.6294,  32.2025,  -0.0000]
   2      0  0.00  [   0.465,    0.029,  169.704]  [   0.000,    0.000,    0.000]  [  1.9622,  -4.2632,  -0.0046]  [  0.0000,   0.0000,   0.0000]
   3      0  0.00  [   0.425,    0.207,   43.208]  [   0.000,    0.000,    0.000]  [  1.2809,   1.0049,  -0.0174]  [  0.0000,   0.0000,   0.0000]
   4      0  0.00  [  -0.247,    0.510,  -13.842]  [   0.000,    0.000,    0.000]  [  0.1467,   0.6694,   0.0221]  [  0.

Download checkpoint (Colab)

In [ ]:
if is_colab():
    from google.colab import files
    files.download("wrench_model_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>